# SH2 superbinder + phosphopeptide: solvated system parameterization

Adapted from `biosensors/PYR1+LCA_solvated_protein+ligand_parameterization.ipynb` (same OpenFF Interchange / GROMACS-export workflow), for a different kind of system: here the "ligand" side is a phosphopeptide, not a small molecule, and the reason for the protein/ligand force-field split is different too -- see below.

In [ ]:
import os
import numpy as np
import nglview
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from openff.toolkit import ForceField, Molecule, Topology
from openff.units import unit
from openff.interchange import Interchange
from openff.interchange.components._packmol import RHOMBIC_DODECAHEDRON, pack_box


## Key decisions

**System:** superbinder SH2 domain (human c-Src SH2, T183V/C188A/K206L) bound to an 11-residue phosphopeptide (`DHEPIYEQWGW`, pTyr at position 6). Starting coordinates from AlphaFold3, validated against the real superbinder crystal structure (PDB 4F5B) in `../af3/align_af3_vs_crystal.py` -- AF3's predicted phosphate position matched the crystal to within 0.9 A, and 100% of real pocket-contact residues (including all three engineered mutations) were also engaged by the AF3-predicted peptide at a 6 A cutoff. That validated pose is preserved exactly through everything below -- no atom that came from AF3 is ever re-embedded or moved.

**Phosphate protonation:** modeled as a **dianion** (both non-bridging, non-carbonyl oxygens deprotonated; net -2 on the phosphate group).

**Peptide termini:** **capped** (ACE on the N-terminus, NME on the C-terminus), not free charged NH3+/COO-.

**Force fields -- why the split differs from the biosensors notebook:** there, the *ligand* got Sage because it's a small molecule with no protein residue template. Here, the phosphopeptide gets Sage for the same underlying reason: `ff14sb_off_impropers` has no residue template for phosphotyrosine (PTR) or for ACE/NME caps, so it cannot parameterize chain B at all. The protein (chain A, all canonical residues) still gets ff14SB. Sage assigns parameters by direct chemical perception (SMIRKS matching), not residue templates, which is exactly why it can handle a non-canonical residue that a protein force field can't.

**A real approximation worth flagging:** ff14SB (protein) and Sage (peptide) were fit independently and are not guaranteed to be perfectly cross-consistent at the protein-peptide interface (nonbonded parameters, in particular). This mirrors exactly what the biosensors protein+ligand notebook already does, so it's a consistent choice for this project, not a new source of error -- but it's a known limitation of the OpenFF Interchange "combine two independently-parameterized pieces" pattern in general, worth remembering if results look off at the interface specifically.

## Inputs

Chain A (protein) and chain B (phosphopeptide) extracted from the AF3 complex via PyMOL.

In [ ]:
af3_cif = "../af3/output/superbinder_sh2_ptyr_peptide/superbinder_sh2_ptyr_peptide_model.cif"
af3_chainA_pdb = "af3_chainA_protein_raw.pdb"
af3_chainB_pdb = "af3_chainB_peptide_raw.pdb"


In [ ]:
# Split the AF3 complex into separate protein / peptide PDBs. (rdkit has no mmCIF reader
# in this version, so this uses PyMOL, which is already used elsewhere in this project.)
!pymol -cq -d "load {af3_cif}, af3; save {af3_chainA_pdb}, af3 and chain A and polymer.protein; save {af3_chainB_pdb}, af3 and chain B"


### Peptide

Capped termini, dianionic phosphotyrosine. Built explicitly with RDKit rather than loaded from a pre-existing SDF (unlike the biosensors ligand), because this exact molecule -- this sequence, this cap chemistry, this protonation state -- doesn't exist anywhere as a file yet; it has to be constructed from the AF3 coordinates.

In [ ]:
peptide_raw = Chem.MolFromPDBBlock(open(af3_chainB_pdb).read(), removeHs=False, sanitize=False)
Chem.SanitizeMol(peptide_raw)

def find_atom(mol, resnum, name):
    for atom in mol.GetAtoms():
        info = atom.GetPDBResidueInfo()
        if info and info.GetResidueNumber() == resnum and info.GetName().strip() == name:
            return atom.GetIdx()
    raise ValueError(f"atom resnum={resnum} name={name} not found")


In [ ]:
# RDKit's default PDB sanitizer gets the PTR phosphate wrong: with no residue template
# for phosphotyrosine, it fills P's valence with a spurious P-H bond instead of a proper
# P=O double bond, and only charges one oxygen (verified interactively before writing this
# -- see conversation). Fix explicitly using AF3's actual atom names (P, O1P, O2P, O3P):
# P=O (double bond) + two P-O(-1) => dianion, net -2 on the phosphate.
rw = Chem.RWMol(peptide_raw)

p_idx, o1p_idx, o2p_idx, o3p_idx = (find_atom(rw, 6, n) for n in ("P", "O1P", "O2P", "O3P"))
for idx in (p_idx, o1p_idx, o2p_idx, o3p_idx):
    rw.GetAtomWithIdx(idx).SetFormalCharge(0)
    rw.GetAtomWithIdx(idx).SetNoImplicit(False)
rw.GetBondBetweenAtoms(p_idx, o3p_idx).SetBondType(Chem.BondType.DOUBLE)
rw.GetAtomWithIdx(o1p_idx).SetFormalCharge(-1)
rw.GetAtomWithIdx(o2p_idx).SetFormalCharge(-1)
for idx in (p_idx, o1p_idx, o2p_idx, o3p_idx):
    rw.GetAtomWithIdx(idx).SetNoImplicit(True)


In [ ]:
# Remove the free C-terminal carboxylate oxygen (OXT) FIRST -- AddAtom() appends new
# atoms at the end of the index list, so removing an existing atom afterward would
# silently shift every subsequently-added atom's index.
try:
    rw.RemoveAtom(find_atom(rw, 11, "OXT"))
except ValueError:
    pass  # no OXT present

n_fixed = rw.GetNumAtoms()  # everything below this index has an AF3-derived coordinate

# N-terminal ACE cap: CH3-C(=O)- bonded to residue 1's backbone N (was free NH2)
n_idx = find_atom(rw, 1, "N")
ace_c, ace_o, ace_ch3 = (rw.AddAtom(Chem.Atom(e)) for e in ("C", "O", "C"))
rw.AddBond(n_idx, ace_c, Chem.BondType.SINGLE)
rw.AddBond(ace_c, ace_o, Chem.BondType.DOUBLE)
rw.AddBond(ace_c, ace_ch3, Chem.BondType.SINGLE)
rw.GetAtomWithIdx(n_idx).SetNoImplicit(True)
rw.GetAtomWithIdx(n_idx).SetNumExplicitHs(1)

# C-terminal NME cap: -NH-CH3 bonded to residue 11's backbone C (was free COO-)
c_idx = find_atom(rw, 11, "C")
nme_n, nme_ch3 = (rw.AddAtom(Chem.Atom(e)) for e in ("N", "C"))
rw.AddBond(c_idx, nme_n, Chem.BondType.SINGLE)
rw.AddBond(nme_n, nme_ch3, Chem.BondType.SINGLE)
rw.GetAtomWithIdx(nme_n).SetNoImplicit(True)
rw.GetAtomWithIdx(nme_n).SetNumExplicitHs(1)

peptide = rw.GetMol()
Chem.SanitizeMol(peptide)
print("Net formal charge:", Chem.GetFormalCharge(peptide))
print("Formula:", rdMolDescriptors.CalcMolFormula(peptide))
# Asp(-1) + Glu(-1)x2 + pTyr dianion(-2), His/others neutral, capped (neutral) termini
assert Chem.GetFormalCharge(peptide) == -5, "unexpected net charge -- check residue protonation assumptions"


In [ ]:
# The 5 new cap atoms have no coordinates yet. Use distance-geometry embedding with a
# coordMap that FIXES every AF3-derived atom in place -- this finds a chemically
# reasonable, non-clashing geometry for the caps only, without disturbing the
# AF3/crystal-validated pose of anything else.
coord_map = {i: peptide.GetConformer().GetAtomPosition(i) for i in range(n_fixed)}
cid = AllChem.EmbedMolecule(peptide, coordMap=coord_map, useRandomCoords=True, randomSeed=42, maxAttempts=200)
assert cid != -1, "cap embedding failed -- try a different randomSeed"

coords = peptide.GetConformer().GetPositions()

max_drift = max(
    np.linalg.norm(coords[i] - np.array([coord_map[i].x, coord_map[i].y, coord_map[i].z]))
    for i in range(n_fixed)
)
print("Max drift of AF3-derived atoms (should be ~0):", round(max_drift, 4), "A")
assert max_drift < 1e-3, "embedding moved atoms it should have held fixed"

min_dist = min(
    np.linalg.norm(coords[i] - coords[j])
    for i in range(len(coords)) for j in range(i + 1, len(coords))
    if not peptide.GetBondBetweenAtoms(i, j)
)
print("Closest non-bonded heavy-atom distance:", round(min_dist, 2), "A")
assert min_dist > 1.5, "cap placement clashes with an existing atom -- try a different randomSeed"


In [ ]:
peptide = Chem.AddHs(peptide, addCoords=True)
peptide_offmol = Molecule.from_rdkit(peptide, allow_undefined_stereo=True)
peptide_offmol.name = "pTyr_peptide"

# The AF3-derived heavy atoms carry residue metadata (ASP/HIS/.../PTR) from the PDB
# they came from; the 5 new cap atoms and all AddHs()-added hydrogens don't --
# GROMACS export requires ALL atoms in a molecule to agree on having (or not having)
# residue names, so give every atom the same flat residue name/number, treating the
# whole capped peptide as one residue -- consistent with how the ligand is treated
# as a single flat molecule in the biosensors notebook this is based on.
for atom in peptide_offmol.atoms:
    atom.metadata["residue_name"] = "LIG"
    atom.metadata["residue_number"] = "1"

print("OpenFF peptide total charge:", peptide_offmol.total_charge)
peptide_offmol.visualize(backend="nglview")


In [ ]:
# Sage's default charge method (AM1BCC) is not designed for molecules this large --
# openff-toolkit itself warns above ~150 atoms it may take hours or hang. Verified
# interactively: a real AM1BCC call on this 208-atom (with H) peptide did not finish
# in 30+ minutes. Use NAGL instead -- a fast GNN-based charge model appropriate for
# peptide-sized molecules (<1s here) -- then hand the precomputed charges to
# Interchange via charge_from_molecules so it doesn't try (and hang) computing AM1BCC.
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

peptide_offmol.assign_partial_charges(
    partial_charge_method="openff-gnn-am1bcc-1.0.0.pt",
    toolkit_registry=NAGLToolkitWrapper(),
)
charge_sum = sum(q.m for q in peptide_offmol.partial_charges)
print(f"Sum of NAGL partial charges: {charge_sum:.6f} (should equal {peptide_offmol.total_charge})")
assert abs(charge_sum - peptide_offmol.total_charge.m) < 1e-3


In [ ]:
peptide_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=[peptide_offmol],
    charge_from_molecules=[peptide_offmol],  # use the NAGL charges assigned above, skip AM1BCC
)


### Protein

In [ ]:
# Add missing atoms/hydrogens at pH 7.5 (matches ../../biosensors/fix_pdb.ipynb).
fixer = PDBFixer(filename=af3_chainA_pdb)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.5)

protein_fixed_pdb = "af3_chainA_protein_fixed_H.pdb"
with open(protein_fixed_pdb, "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)


In [ ]:
protein_full = Topology.from_pdb(protein_fixed_pdb)
protein = protein_full.molecule(0)
protein.name = "protein"
view_protein = protein.visualize(backend="nglview")
view_protein.clear_representations()
view_protein.add_representation("cartoon", selection="protein")
view_protein


In [ ]:
ff14sb = ForceField("ff14sb_off_impropers_0.0.3.offxml")
protein_intrcg = Interchange.from_smirnoff(
    force_field=ff14sb,
    topology=protein.to_topology(),
)


### Dock peptide + protein

In [ ]:
docked_intrcg = protein_intrcg.combine(peptide_intrcg)

w = docked_intrcg.visualize()
w.clear_representations()
w.add_representation(
    "cartoon",
    radius=0.1,
    selection=[*range(protein_intrcg.topology.n_atoms)],
)
w.add_representation(
    "licorice",
    selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)],
)
w


Check partial charge of protein+peptide complex

In [ ]:
total_charge = round(sum(docked_intrcg["Electrostatics"].charges.values()), 3)

assert total_charge == protein.total_charge + peptide_offmol.total_charge, (
    f"Total charge of the system is {total_charge}, not {protein.total_charge + peptide_offmol.total_charge}"
)
total_charge


### Solvent

In [ ]:
total_charge_e = float(total_charge.m)

if total_charge_e < 0:
    counterion = Molecule.from_smiles("[Na+]")
    counterion.name = "NA"
    n_counterions = int(round(abs(total_charge_e)))
elif total_charge_e > 0:
    counterion = Molecule.from_smiles("[Cl-]")
    counterion.name = "CL"
    n_counterions = int(round(abs(total_charge_e)))
else:
    counterion = None
    n_counterions = 0

water = Molecule.from_smiles("O")
water.name = "SOL"
water.generate_conformers(n_conformers=1)

print(f"Net charge: {total_charge_e}, counterion: {counterion.name if counterion else None} x {n_counterions}")


In [ ]:
box_shape = "dodecahedron"

xyz = protein.conformers[0].to(unit.nanometer).m  # (N, 3) in nm, unitless array
centroid = xyz.mean(axis=0)
protein_radius_nm = np.sqrt(((xyz - centroid) ** 2).sum(axis=1).max())

buffer_nm = 2.0  # distance from edge to center
scale_nm = 2.0 * protein_radius_nm + buffer_nm  # "box length" of unit cell

box_vectors = (scale_nm * RHOMBIC_DODECAHEDRON) * unit.nanometer

box_nm = box_vectors.to(unit.nanometer).m
V_box_nm3 = abs(np.linalg.det(box_nm))
V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)  # crude spherical estimate
waters_per_nm3 = 33.4  # ~1 g/mL at 300 K
n_water = int(waters_per_nm3 * max(V_box_nm3 - V_solute_nm3, 0.0))

print(f"Protein radius: {protein_radius_nm:.3f} nm")
print(f"Dodecahedron scale: {scale_nm:.3f} nm")
print(f"Box volume: {V_box_nm3:.2f} nm^3")
print(f"Packing waters: {n_water}, counterions: {n_counterions}")

molecules = [water]
number_of_copies = [n_water]
if counterion is not None and n_counterions > 0:
    molecules.append(counterion)
    number_of_copies.append(n_counterions)

packed_topology = pack_box(
    solute=docked_intrcg.topology,
    molecules=molecules,
    number_of_copies=number_of_copies,
    box_vectors=box_vectors,
    center_solute=True,
    tolerance=2.0 * unit.angstrom,
    working_directory="packmol_solv_dodecahedron",
    retain_working_files=True,
)

print("Total molecules packed:", packed_topology.n_molecules)
print("Box vectors (nm):\n", packed_topology.box_vectors.to(unit.nanometer))


In [ ]:
packed_topology.to_file(f"packed_{box_shape}_sh2_superbinder_pTyr.pdb")
nglview.show_structure_file(f"packed_{box_shape}_sh2_superbinder_pTyr.pdb")


In [ ]:
topology_molecules = [water] * n_water
if counterion is not None:
    topology_molecules += [counterion] * n_counterions


In [ ]:
water_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=topology_molecules,
)


### Put everything together
Dock protein+peptide complex with the solvent

In [ ]:
system_intrcg = docked_intrcg.combine(water_intrcg)
system_intrcg.positions = packed_topology.get_positions()
system_intrcg.box = packed_topology.box_vectors


In [ ]:
w = system_intrcg.visualize()
w.clear_representations()
w.add_unitcell()
w.add_representation("licorice", radius=0.2, selection=[*range(protein_intrcg.topology.n_atoms)])
w.add_representation("spacefill", selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)])
w.add_representation("licorice", radius=0.1, selection=[*range(docked_intrcg.topology.n_atoms, system_intrcg.topology.n_atoms)])
w.center()
w


## Export to GROMACS

In [ ]:
os.makedirs("gromacs", exist_ok=True)
os.chdir("gromacs")
os.getcwd()


In [ ]:
# No hydrogen mass repartitioning -- standard masses (1.007947 amu, openff-interchange's
# own default), paired with a standard 2 fs production timestep in prod_md.mdp.
system_intrcg.to_gromacs(prefix="sh2_superbinder_pTyr_dodecahedron", decimal=3, monolithic=False)
